In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "CLAUDE.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

path = PROJECT_ROOT / "data" / "raw" / "statcast_2024-04-15_ingested_2026-08-31.parquet"
df = pd.read_parquet(path)
print(df.shape)

(4362, 119)


In [2]:
for c in ["game_pk", "at_bat_number", "pitch_number", "inning"]:
    print(f"{c}: {df[c].nunique()} unique, dtype={df[c].dtype}")

print()
print("games:", df["game_pk"].nunique())
print("plate appearances:", df.groupby(["game_pk", "at_bat_number"]).ngroups)
print("pitches:", len(df))

game_pk: 15 unique, dtype=Int64
at_bat_number: 86 unique, dtype=Int64
pitch_number: 11 unique, dtype=Int64
inning: 11 unique, dtype=Int64

games: 15
plate appearances: 1111
pitches: 4362


In [3]:
df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)
print(df[["game_pk", "at_bat_number", "pitch_number", "balls", "strikes", "description"]].head(15))

    game_pk  at_bat_number  pitch_number  balls  strikes      description
0    744951              1             1      0        0    called_strike
1    744951              1             2      0        1    hit_into_play
2    744951              2             1      0        0    called_strike
3    744951              2             2      0        1    called_strike
4    744951              2             3      0        2    hit_into_play
5    744951              3             1      0        0    called_strike
6    744951              3             2      0        1             ball
7    744951              3             3      1        1             ball
8    744951              3             4      2        1  swinging_strike
9    744951              3             5      2        2    called_strike
10   744951              4             1      0        0   automatic_ball
11   744951              4             2      1        0             ball
12   744951              4            

In [4]:
first_pitches = df[df["pitch_number"] == 1]
print(first_pitches[["balls", "strikes"]].value_counts())

balls  strikes
0      0          1111
Name: count, dtype: int64


In [5]:
pa_end = df[df["events"].notna()]
print(len(pa_end))
print()
print(pa_end["events"].value_counts())

1110

events
field_out                    494
strikeout                    234
single                       153
walk                          87
double                        47
force_out                     22
grounded_into_double_play     20
home_run                      20
hit_by_pitch                   7
sac_fly                        6
sac_bunt                       5
truncated_pa                   3
intent_walk                    3
catcher_interf                 3
triple                         2
fielders_choice                2
field_error                    1
double_play                    1
Name: count, dtype: int64


In [6]:
last_pitch_idx = df.groupby(["game_pk", "at_bat_number"])["pitch_number"].idxmax()
last_pitches = df.loc[last_pitch_idx]

print("last pitches:", len(last_pitches))
print("with events:", last_pitches["events"].notna().sum())
print("without events:", last_pitches["events"].isna().sum())

last pitches: 1111
with events: 1110
without events: 1


In [7]:
orphan = last_pitches[last_pitches["events"].isna()]
print(orphan[["game_pk", "inning", "inning_topbot", "at_bat_number",
              "pitch_number", "balls", "strikes", "outs_when_up",
              "description", "on_1b", "on_2b", "on_3b"]].to_string())

      game_pk  inning inning_topbot  at_bat_number  pitch_number  balls  strikes  outs_when_up   description   on_1b  on_2b  on_3b
2107   746080       6           Top             44             3      0        2             2  blocked_ball  664774   <NA>   <NA>
